## Setting up model

In [2]:
from langchain.chat_models import ChatOpenAI

chat = ChatOpenAI(model_name="gpt-5-nano", temperature=1)


## Chatting with Message Objects

In [3]:
# from langchain.schema import HumanMessage, AIMessage, SystemMessage

# messages = [
#     SystemMessage(content="You are a geography expert and you only reply in Italian."),
#     AIMessage(content="Ciao, mi chiamo Palol!"),
#     HumanMessage(content="What is the distance between Mexico and Thailand. Also, what is your name?")
# ]

# chat.predict_messages(messages)

## Using ChatPromptTemplate

In [4]:
from langchain.prompts import PromptTemplate, ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a geography expert and you only reply in {language}."),
        ("ai", "Ciao, mi chiamo {name}!"),
        ("human", "hat is the distance between {country_a} and {country_b}. Also, what is your name?")
    ]
)

prompt = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand"
)

chat.predict_messages(prompt)

AIMessage(content='Απόσταση Μεξικό ↔ Ταϊλάνδη (ευθεία, μεγάλο κύκλο) για παράδειγμα Μεξικό Σίτι → Μπανγκόκ: περίπου 15,8 χιλιάδες χιλιόμετρα (περίπου 9,8 χιλιάδες μίλια). Η ακριβής απόσταση εξαρτάται από τα σημεία εκκίνησης/προσγείωσης που επιλέγετε.\n\nΤο όνομά μου: ChatGPT. Μπορείς να με αποκαλείς και Γεωγράφος.')

## OutputParser

In [12]:
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items))

In [13]:
template = ChatPromptTemplate.from_messages([
    ("system","You are a list generating machine. Everything you are asked will be answered with a comma separated list of max {max_items} in lowercase. Do NOT reply with anything else."),
    ("human", "{question}")
])

prompt = template.format_messages(
    max_items=10,
    question="What are the planets?"
)

result = chat.predict_messages(prompt)

p = CommaOutputParser()

p.parse(result.content)

['mercury', 'venus', 'earth', 'mars', 'jupiter', 'saturn', 'uranus', 'neptune']

## LCEL

In [16]:
chain = template | chat | CommaOutputParser()

chain.invoke({
    "max_items":5,
    "question":"What are the pokemons?"
})

['pikachu', 'bulbasaur', 'charizard', 'squirtle', 'eevee']

## Chaining Chains

In [25]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to find ingredients."),
    ("human", "I want to cook {cuisine} food.")
])

chef_chain = chef_prompt | chat

In [ ]:
veg_chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a vegetarian chef specialized on making traditional recipies vegetarian. You find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it."),
    ("human", "{recipe}")
])

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe": chef_chain} | veg_chain

final_chain.invoke({"cuisine": "indian"})


Fantastic! Indian cooking is all about bold flavors and fresh ingredients. I can tailor to what you have, but here are three easy, beginner-friendly options (vegetarian and meat-friendly) plus a quick rice side to pair with everything.

Option 1: Chana Masala (Chickpea Curry) — quick, hearty, vegan
Ingredients (serves 2-3)
- 1 tbsp oil
- 1 medium onion, finely chopped
- 2–3 garlic cloves, minced
- 1 inch ginger, minced
- 1 green chili, optional, minced
- 1 tsp cumin seeds
- 1 cup canned crushed tomatoes (or 2 chopped tomatoes)
- 1 tsp ground coriander
- 1/2 tsp turmeric
- 1/2 tsp chili powder or paprika (adjust to taste)
- 1 tsp garam masala
- 1 can (15 oz) chickpeas, drained and rinsed
- 1/2 cup water (as needed)
- Salt to taste
- Fresh cilantro, chopped (for garnish)

Steps
1) Heat oil in a pan. Add cumin seeds and let them crackle 10–20 seconds.
2) Add onion; sauté until soft and translucent.
3) Stir in garlic, ginger, and chili; cook 1 minute.
4) Add coriander, turmeric, and chili 

AIMessageChunk(content='Love it. I can tailor these to what you have and your spice tolerance, without changing the spirit of the recipes.\n\nA few quick questions to tailor perfectly:\n- Are you strictly vegetarian, or ok with dairy (paneer/yogurt/ghee) or meat (poultry/beef) additions?\n- Spice level: mild, medium, or hot?\n- What do you actually have in your pantry? (Protein options like chickpeas, lentils, paneer; vegetables like onion, tomato, garlic, ginger, potatoes, cauliflower; staples like basmati rice, canned tomatoes, coconut milk; spices like cumin, coriander, turmeric, garam masala, chili powder.)\n- Any dietary restrictions (no onion/garlic, gluten-free, etc.)?\n\nIf you’d like, I can craft a single exact recipe right away based on common pantry items. For example, here’s a precise Chana Masala you can use as-is (serves 2–3):\n\nChana Masala (Chickpea Curry) — exact, serves 2–3\n- Oil: 15 ml (1 tablespoon)\n- Onion: 150 g, finely chopped\n- Garlic: 2–3 cloves, minced (ab

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

haiku_prompt = ChatPromptTemplate.from_messages([
    ("system", "You make a haiku about programming languages"),
    ("human", "Write a haiku about this programming language: {language}")
])

haiku_chain = haiku_prompt | chat

haiku_explainer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain the meaning, imagery, and references of the given haiku."),
    ("human", "Here is a haiku: {haiku}. Explain this haiku.")
])

haiku_explainer_chain = haiku_explainer_prompt | chat

final_chain = {"haiku": haiku_chain} | haiku_explainer_chain

final_chain.invoke({"language": "Python"})

Python slithers fast,
Code with simplicity pure,
Snakes make data dance.This haiku is a clever play on words, using the image of a python as both a snake and a programming language. In the first line, "Python slithers fast," the image of a snake moving quickly conjures a sense of speed and agility, which can be likened to how efficiently Python code can be executed. The second line, "Code with simplicity pure," suggests that writing code in Python is concise and straightforward, as the language is known for its readability and simplicity. Finally, the third line, "Snakes make data dance," depicts the idea that Python, like a snake, has the ability to manipulate and interact with data in a dynamic and fluid manner. Overall, the haiku conveys the idea that Python is a versatile and powerful tool for working with data in a streamlined and elegant way.

AIMessageChunk(content='This haiku is a clever play on words, using the image of a python as both a snake and a programming language. In the first line, "Python slithers fast," the image of a snake moving quickly conjures a sense of speed and agility, which can be likened to how efficiently Python code can be executed. The second line, "Code with simplicity pure," suggests that writing code in Python is concise and straightforward, as the language is known for its readability and simplicity. Finally, the third line, "Snakes make data dance," depicts the idea that Python, like a snake, has the ability to manipulate and interact with data in a dynamic and fluid manner. Overall, the haiku conveys the idea that Python is a versatile and powerful tool for working with data in a streamlined and elegant way.')

## FewShotPromptTemplate

### LengthBased

In [58]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts.example_selector import LengthBasedExampleSelector
from langchain.prompts.example_selector.base import BaseExampleSelector

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

examples = [
{
"question": "What do you know about France?",
"answer": """
Here is what I know:
Capital: Paris
Language: French
Food: Wine and Cheese
Currency: Euro
""",
},
{
"question": "What do you know about Italy?",
"answer": """
I know this:
Capital: Rome
Language: Italian
Food: Pizza and Pasta
Currency: Euro
""",
},
{
"question": "What do you know about Greece?",
"answer": """
I know this:
Capital: Athens
Language: Greek
Food: Souvlaki and Feta Cheese
Currency: Euro
""",
},
]

class RandomExampleSelector(BaseExampleSelector):

    def __init__(self, examples):
        self.examples = examples
    
    def add_example(self, example):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)]

example_template = """
    Human: {question}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

# example_selector = LengthBasedExampleSelector(
#     examples=examples,
#     example_prompt=example_prompt,
#     max_length=180
# )

example_selector = RandomExampleSelector(
    examples=examples
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"]
)

prompt.format(country="Brazil")

# chain = prompt | chat

# chain.invoke({
#     "country": "Turkey"
# })



'\n    Human: What do you know about Italy?\n    AI: \nI know this:\nCapital: Rome\nLanguage: Italian\nFood: Pizza and Pasta\nCurrency: Euro\n\n\n\nHuman: What do you know about Brazil?'

In [ ]:
	
from langchain.chat_models import ChatOpenAI
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import ChatMessagePromptTemplate, ChatPromptTemplate

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)


examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]


example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "What do you know about {country}?"),
        ("ai", "{answer}"),
    ]
)

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a geography expert, you give short answers."),
        example_prompt,
        ("human", "What do you know about {country}?"),
    ]
)

chain = final_prompt | chat

chain.invoke({"country": "Thailand"})

In [37]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

examples = [
{
"question": "Tell me about the movie Inception",
"answer": """
Here is information about the movie:
Title: Inception
Director: Christopher Nolan
Main Cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy
Budget: $160 million
Box Office: $836 million
Genre: Science Fiction, Action, Thriller
Synopsis: A skilled thief who steals corporate secrets through dream-sharing technology is given the task of planting an idea in a target’s subconscious.
"""
},
{
"question": "Tell me about the movie Titanic",
"answer": """
Here is information about the movie:
Title: Titanic
Director: James Cameron
Main Cast: Leonardo DiCaprio, Kate Winslet
Budget: $200 million
Box Office: $2.2 billion
Genre: Romance, Drama
Synopsis: A tragic love story unfolds aboard the RMS Titanic as two young passengers from different social classes fall in love before the ship's doomed maiden voyage.
"""
},
{
"question": "Tell me about the movie The Matrix",
"answer": """
Here is information about the movie:
Title: The Matrix
Director: The Wachowskis
Main Cast: Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss
Budget: $63 million
Box Office: $467 million
Genre: Science Fiction, Action
Synopsis: A computer hacker discovers that reality is a simulated world controlled by machines and joins a rebellion to free humanity.
"""
},
]

example_template = """
Human: {question}
AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="Human: Tell me about the movie {movie}",
    input_variables=["movie"]
)

chain = prompt | chat

chain.invoke({
    "movie": "Interstellar"
})

Here is information about the movie:
Title: Interstellar
Director: Christopher Nolan
Main Cast: Matthew McConaughey, Anne Hathaway, Jessica Chastain, Michael Caine
Budget: $165 million
Box Office: $677 million
Genre: Science Fiction, Adventure, Drama
Synopsis: In a future where Earth is becoming uninhabitable, a former NASA pilot joins a team of explorers through a wormhole to seek a new home for humanity while facing time dilation and the vast distances of space.

AIMessageChunk(content='Here is information about the movie:\nTitle: Interstellar\nDirector: Christopher Nolan\nMain Cast: Matthew McConaughey, Anne Hathaway, Jessica Chastain, Michael Caine\nBudget: $165 million\nBox Office: $677 million\nGenre: Science Fiction, Adventure, Drama\nSynopsis: In a future where Earth is becoming uninhabitable, a former NASA pilot joins a team of explorers through a wormhole to seek a new home for humanity while facing time dilation and the vast distances of space.')

## Serialization and Composition

In [61]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

# prompt = load_prompt("./prompt.json")
prompt = load_prompt("./prompt.yaml")

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

prompt.format(country="Germany")


'What is the capital of Germany?'

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts.pipeline import PipelinePromptTemplate

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

intro = PromptTemplate.from_template(
"""
You are a role playing assistant.
And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
"""
This is an example of how you talk:

Human: {example_question}
You: {example_answer}
"""
)

start = PromptTemplate.from_template(
"""
Start now!

Human: {question}
You:
"""
)

final = PromptTemplate.from_template(
"""
{intro}

{example}

{start}
"""
)

prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start)
]

full_prompt = PipelinePromptTemplate(
    final_prompt=final,
    pipeline_prompts=prompts
)

# full_prompt.format(
#     character="Pirate",
#     example_question="What is your location?",
#     example_answer="Arrrg! That is a secret!!",
#     question="What is your favorite food?"
# )

chain = full_prompt | chat

chain.invoke({
    "character":"Pirate",
    "example_question":"What is your location?",
    "example_answer":"Arrrg! That is a secret!!",
    "question":"What is your favorite food?"
})


Arrr, matey! Me favorite food be salt pork with hardtack, washed down with a hearty swig o’ grog. Keeps this old sea dog ready for the next raid, aye!

AIMessageChunk(content='Arrr, matey! Me favorite food be salt pork with hardtack, washed down with a hearty swig o’ grog. Keeps this old sea dog ready for the next raid, aye!')

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback

chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

with get_openai_callback() as usage:
    a = chat.predict("What is the recipe for soju?")
    b = chat.predict("What is the recipe for bread?")
    print(a, b, "\n")
    print(usage)

Soju doesn’t have a single universal recipe. It’s a category of Korean distilled (and often diluted) spirits that can be made in different ways. Here’s a safe, high-level overview of how traditional and modern soju is made:

- Base ingredients:
  - Traditional: starch sources such as rice (most common), barley, or wheat.
  - Modern/industrial: a neutral base spirit can be made from various starches (rice, corn, potatoes, sweet potatoes, etc.) and then diluted.

- Fermentation starter:
  - Traditional soju uses nuruk (a mold- and yeast-containing fermentation starter).
  - Many modern versions use commercial yeast strains.

- Fermentation:
  - The starch is converted to sugars and then fermented to alcohol in a mash.

- Distillation:
  - The fermented mash is distilled (usually in a pot still) to concentrate the ethanol.
  - The “hearts” portion is collected; heads and tails are typically discarded or purified/distilled separately.

- Dilution and finishing:
  - The distilled spirit is 

## Caching

In [71]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.globals import set_llm_cache, set_debug
from langchain.cache import InMemoryCache, SQLiteCache

# set_llm_cache(InMemoryCache())
# set_debug(True)

set_llm_cache(SQLiteCache("cache.db"))


chat = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1,
    # streaming=True,
    # callbacks=[StreamingStdOutCallbackHandler()]
)

chat.predict("how do you make italian pasta")


[llm/start] [1:llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: how do you make italian pasta"
  ]
}
[llm/end] [1:llm:ChatOpenAI] [16.79s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Here’s a simple guide to making Italian pasta, covering both dried pasta and fresh pasta from scratch.\n\n1) Dried pasta (store-bought)\n- What you need: plenty of water, salt, pasta, your sauce.\n- Steps:\n  - Bring a large pot of water to a rolling boil. Salt it generously (about 1–2 tablespoons of salt per 4–6 liters of water).\n  - Add the pasta and stir to prevent sticking.\n  - Cook until al dente according to the package instructions.\n  - Reserve a little pasta water (about 1/2 cup) in case you need to loosen the sauce.\n  - Drain and toss with your sauce. Add a splash of pasta water to help emulsify the sauce if needed.\n  - Finish with grated cheese if you like.\n\n2) Fresh pasta from scratch (egg pasta)\n- For 2 servings (adjust as ne

'Here’s a simple guide to making Italian pasta, covering both dried pasta and fresh pasta from scratch.\n\n1) Dried pasta (store-bought)\n- What you need: plenty of water, salt, pasta, your sauce.\n- Steps:\n  - Bring a large pot of water to a rolling boil. Salt it generously (about 1–2 tablespoons of salt per 4–6 liters of water).\n  - Add the pasta and stir to prevent sticking.\n  - Cook until al dente according to the package instructions.\n  - Reserve a little pasta water (about 1/2 cup) in case you need to loosen the sauce.\n  - Drain and toss with your sauce. Add a splash of pasta water to help emulsify the sauce if needed.\n  - Finish with grated cheese if you like.\n\n2) Fresh pasta from scratch (egg pasta)\n- For 2 servings (adjust as needed):\n  - 250 g (about 2 cups) all-purpose flour or 00 flour\n  - 2–3 large eggs (or 1 egg per 100 g flour)\n- Steps:\n  - On a clean surface, mound the flour and make a well in the center. Crack in the eggs.\n  - Beat the eggs with a fork an

In [70]:
chat.predict("how do you make italian pasta")

[llm/start] [1:llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: how do you make italian pasta"
  ]
}
[llm/end] [1:llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Do you want to make fresh pasta from scratch or just cook store-bought dried pasta? Here are both quick, easy options.\n\nFresh pasta dough (egg pasta) from scratch\n- Ingredients (per 2 servings): 200 g flour (preferably 00 flour) or a mix of 150 g 00 + 50 g all-purpose, 2 large eggs, pinch of salt. Optional: 1 tablespoon olive oil.\n- Ratios: a common rule is 1 large egg per 100 g flour (or 2 eggs for 200 g flour).\n- Tools: clean work surface, rolling pin or pasta machine, bench scraper, plastic wrap.\n- Steps:\n  1) Pile flour on the counter and make a well in the middle. Crack eggs into the well and add a pinch of salt.\n  2) Beat the eggs with a fork, gradually drawing in flour from the edges until a rough dough forms.\n  3) Knead on the counter

'Do you want to make fresh pasta from scratch or just cook store-bought dried pasta? Here are both quick, easy options.\n\nFresh pasta dough (egg pasta) from scratch\n- Ingredients (per 2 servings): 200 g flour (preferably 00 flour) or a mix of 150 g 00 + 50 g all-purpose, 2 large eggs, pinch of salt. Optional: 1 tablespoon olive oil.\n- Ratios: a common rule is 1 large egg per 100 g flour (or 2 eggs for 200 g flour).\n- Tools: clean work surface, rolling pin or pasta machine, bench scraper, plastic wrap.\n- Steps:\n  1) Pile flour on the counter and make a well in the middle. Crack eggs into the well and add a pinch of salt.\n  2) Beat the eggs with a fork, gradually drawing in flour from the edges until a rough dough forms.\n  3) Knead on the counter for about 8–10 minutes until smooth and elastic. If very sticky, dust with a little flour; if too dry, sprinkle with a few drops of water or an extra splash of egg.\n  4) Wrap the dough in plastic and rest 30–60 minutes at room temperatu

## Memory

In [8]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)

memory.save_context({"input": "Hi"}, {"output": "How are you?"})

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi'), AIMessage(content='How are you?')]}

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=4
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

# buffer
add_message(1, 1)
add_message(2, 2)
add_message(3, 3)
add_message(4, 4)
add_message(5, 5)

memory.load_memory_variables({})

{'history': [HumanMessage(content='2'),
  AIMessage(content='2'),
  HumanMessage(content='3'),
  AIMessage(content='3'),
  HumanMessage(content='4'),
  AIMessage(content='4'),
  HumanMessage(content='5'),
  AIMessage(content='5')]}

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryMemory

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=1
)

memory = ConversationSummaryMemory(llm=llm)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    return memory.load_memory_variables({})

# summary
add_message("Hi I live in USA", "Hi! Nice to meet you.")
add_message("What is the weather in Ann Arbor?", "It is raining")

get_history()



{'history': 'The human says they live in the USA, and the AI responds with a friendly greeting: "Hi! Nice to meet you." The human asks about the weather in Ann Arbor, and the AI replies that it is raining.'}

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory

llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=1
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=20,
    return_messages=True
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    return memory.load_memory_variables({})

# buffer + summary
add_message("Hi I live in USA", "Hi! Nice to meet you.")
get_history()
add_message("What is the weather in Ann Arbor?", "It is raining")
get_history()


{'history': [SystemMessage(content='The human mentions that they live in the USA. The AI responds with a polite greeting. The human then asks about the weather in Ann Arbor.'),
  AIMessage(content='It is raining')]}

In [3]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import LLMChain
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    tiktoken_model_name="gpt-3.5-turbo",
    temperature=1
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=80,
    memory_key="chat_history",
    return_messages=True
)

# template = """

#     You are a helpful AI talking to a human.

#     {chat_history}
#     Human:{question}
#     You:
# """

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to a human"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# chain = LLMChain(
#     llm=llm,
#     memory=memory,
#     # prompt=PromptTemplate.from_template(template),
#     prompt=prompt,
#     verbose=True
# )

# chain.predict(question="My name is WJ")
def load_memory(_):
    return memory.load_memory_variables({})["chat_history"]

chain = RunnablePassthrough.assign(chat_history=load_memory) | prompt | llm

def invoke_chain(question):
    result= chain.invoke({
    "question": "My name is WJ"
    })
    memory.save_context({"input": question}, {"output": result.content})
    print(result)


In [4]:
# chain.predict(question="I live in Seoul")
invoke_chain("I live in Seoul")

content='Hi WJ! Nice to meet you. Do you prefer to be called WJ, or would you like a different name? How can I help you today—questions, writing, planning, brainstorming, or something else? I can remember your name for this chat if you’d like.'


In [7]:
# chain.predict(question="What is my name?")
invoke_chain("What is my name?")

content='Nice to meet you, WJ. I’ll remember your name for this chat. How can I help today? A few ideas:\n- Practice Korean or translate something\n- Plan activities or get tips for living in Seoul\n- Help with writing (emails, essays, resumes)\n- Brainstorm ideas or study plans\n- Answer questions about Korea or daily life Here’s where you want to start?'


In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import LLMChain
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate

llm = ChatOpenAI(
    model_name="gpt-5-nano",
    tiktoken_model_name="gpt-3.5-turbo",
    temperature=1
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100,
    memory_key="chat_history",
    return_messages=True
)

examples = [
    {"input": "Top Gun", "output": "🛩️👨‍✈️🔥"},
    {"input": "The Godfather", "output": "👨‍👨‍👦🔫🍝"}
]

few_shot = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("ai", "{output}")
    ])
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You will be given a movie title. Reply with EXACTLY three emojis that represent the movie."),
    few_shot,
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

def load_memory(_):
    return memory.load_memory_variables({})["chat_history"]

chain = RunnablePassthrough.assign(chat_history=load_memory) | prompt | llm

def invoke_chain(question):
    result= chain.invoke({
    "question": question
    })
    memory.save_context({"input": question}, {"output": result.content})
    print(result.content)

invoke_chain("Tron")
invoke_chain("Whiplash")


💻👾🕹️
🥁🎶👨‍🏫


In [2]:
recall_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on the chat history. Reply with the FIRST movie title the user asked about, only."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

recall_chain = RunnablePassthrough.assign(chat_history=load_memory) | recall_prompt | llm

result = recall_chain.invoke({"question": "What was the first movie I asked about?"})
print(result.content)

Tron
